<a href="https://colab.research.google.com/github/andretorquato443/Previsao-de-AVC-Ciencia-de-Dados/blob/main/trabalho_final_de_cd_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/andretorquato443/Previsao-de-AVC-Ciencia-de-Dados/refs/heads/main/healthcare-dataset-stroke-data.csv")

import matplotlib.pyplot as plt
import numpy as np # linear algebra
import plotly.express as px
import altair as alt
alt.data_transformers.disable_max_rows()



DataTransformerRegistry.enable('default')

Definição do Problema

O Acidente Vascular Cerebral (AVC) é uma das principais causas de morte e incapacidade no mundo, tornando a identificação precoce de indivíduos de risco uma questão relevante para a área da saúde.

Desse modo , o objetivo deste trabalho é investigar até que ponto características de um paciente podem ser utilizadas para estimar sua probabilidade de sofrer um AVC. Para isso, será utilizado um conjunto de dados contendo informações demográficas, hábitos de vida e condições médicas de diferentes indivíduos.

Como o AVC é um evento relativamente raro na população analisada, o problema apresenta forte desbalanceamento entre pacientes que sofreram AVC e pacientes que não sofreram. Dessa forma, o foco do trabalho não é apenas produzir classificações binárias exatas para cada indivíduo, mas principalmente desenvolver modelos capazes de atribuir níveis de risco coerentes aos pacientes, permitindo distinguir indivíduos com maior probabilidade de AVC daqueles com menor probabilidade.

---------------------------------------------------------------------------------------------------------------------
Importa dataset "Stroke Prediction Dataset" , de fedesoriano e mostra as primeiras linhas da tabela desses dados:


In [8]:
df = pd.read_csv("https://raw.githubusercontent.com/andretorquato443/Previsao-de-AVC-Ciencia-de-Dados/refs/heads/main/healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


Aqui , percebe-se a estrutura do dataset. Nele é possível identificar que apenas a coluna 'bmi' possui alguns valores faltantes.

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


Aqui , é possível analisar as estatísticas descritivas dos dados . Nota-se que apenas aproximadamente 4,9 % dos pacientes para esses dados são identificados como positivos para AVC , comprovando o desbalanceamento dos dados.

In [10]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


VISUALIZAÇÕES :

In [11]:
import pandas as pd
import altair as alt

# ==========================
# Taxa de AVC por fator
# ==========================

factors = {
    "Idosos (60+)": df[df["age"] >= 60]["stroke"].mean() * 100,
    "Hipertensão": df[df["hypertension"] == 1]["stroke"].mean() * 100,
    "Doença Cardíaca": df[df["heart_disease"] == 1]["stroke"].mean() * 100,
    "Glicose Alta": df[df["avg_glucose_level"] >= 126]["stroke"].mean() * 100,
    "Já Fumou": df[df["smoking_status"].isin(["smokes", "formerly smoked"])]["stroke"].mean() * 100
}

plot_df = pd.DataFrame({
    "Fator": factors.keys(),
    "Taxa AVC": factors.values()
})

plot_df = plot_df.sort_values(
    "Taxa AVC",
    ascending=False
)

chart = (
    alt.Chart(plot_df)
    .mark_bar(cornerRadiusTopLeft=6, cornerRadiusTopRight=6)
    .encode(
        x=alt.X(
            "Fator:N",
            sort="-y",
            title=""
        ),
        y=alt.Y(
            "Taxa AVC:Q",
            title="Taxa de AVC (%)"
        ),
        color=alt.Color(
            "Taxa AVC:Q",
            scale=alt.Scale(scheme="orangered"),
            title="Taxa (%)"
        ),
        tooltip=[
            alt.Tooltip("Fator:N"),
            alt.Tooltip(
                "Taxa AVC:Q",
                format=".2f",
                title="Taxa de AVC (%)"
            )
        ]
    )
    .properties(
        width=650,
        height=400,
        title="Incidência de AVC entre Pessoas com Diferentes Fatores de Risco"
    )
)

labels = (
    alt.Chart(plot_df)
    .mark_text(
        dy=-10,
        fontSize=13,
        fontWeight="bold"
    )
    .encode(
        x="Fator:N",
        y="Taxa AVC:Q",
        text=alt.Text(
            "Taxa AVC:Q",
            format=".1f"
        )
    )
)

chart + labels

alt.LayerChart(...)

A Taxa de avc é calculada agrupando pessoas dentro do mesmo fator de risco  e tirando-se a media de casos de AVC por grupo.

Observa-se que indivíduos com doença cardíaca apresentaram a maior incidência de AVC , seguidos por hipertensos  e idosos com mais de 60 anos . Os resultados sugerem que condições cardiovasculares e o envelhecimento estão fortemente associados à ocorrência de AVC no conjunto de dados analisado.

In [12]:

import pandas as pd
import altair as alt

# =====================================
# Construção do score de risco
# =====================================

df_risk = df.copy()

df_risk["ever_smoked"] = (
    df_risk["smoking_status"]
    .isin(["smokes", "formerly smoked"])
    .astype(int)
)

df_risk["high_glucose"] = (
    (df_risk["avg_glucose_level"] >= 126)
    .astype(int)
)

df_risk["risk_score"] = (
    df_risk["hypertension"]
    + df_risk["heart_disease"]
    + df_risk["ever_smoked"]
    + df_risk["high_glucose"]
)

# =====================================
# Taxa de AVC por score
# =====================================

risk_plot = (
    df_risk.groupby("risk_score")
    .agg(
        stroke_rate=("stroke", "mean"),
        count=("stroke", "size")
    )
    .reset_index()
)

risk_plot["stroke_rate"] *= 100

# =====================================
# Linha principal
# =====================================

line = (
    alt.Chart(risk_plot)
    .mark_line(
        strokeWidth=5
    )
    .encode(
        x=alt.X(
            "risk_score:O",
            title="Número de Fatores de Risco"
        ),
        y=alt.Y(
            "stroke_rate:Q",
            title="Taxa de AVC (%)"
        )
    )
)

# =====================================
# Pontos
# =====================================

points = (
    alt.Chart(risk_plot)
    .mark_circle(
        size=220,
        filled=True
    )
    .encode(
        x="risk_score:O",
        y="stroke_rate:Q",
        color=alt.Color(
            "stroke_rate:Q",
            scale=alt.Scale(scheme="reds"),
            title="Taxa de AVC (%)"
        ),
        tooltip=[
            alt.Tooltip(
                "risk_score:O",
                title="Fatores de risco"
            ),
            alt.Tooltip(
                "stroke_rate:Q",
                title="Taxa AVC (%)",
                format=".2f"
            ),
            alt.Tooltip(
                "count:Q",
                title="Pacientes"
            )
        ]
    )
)

# =====================================
# Valores sobre os pontos
# =====================================

labels = (
    alt.Chart(risk_plot)
    .mark_text(
        dy=-15,
        fontSize=13,
        fontWeight="bold"
    )
    .encode(
        x="risk_score:O",
        y="stroke_rate:Q",
        text=alt.Text(
            "stroke_rate:Q",
            format=".1f"
        )
    )
)

# =====================================
# Gráfico final
# =====================================

(line + points + labels).properties(
    width=750,
    height=450,
    title={
        "text": "Impacto do Acúmulo de Fatores de Risco na Incidência de AVC",
        "subtitle": [
            "Hipertensão + Doença Cardíaca + Tabagismo + Glicose Elevada"
        ]
    }
)


alt.LayerChart(...)

A Taxa de avc é calculada agrupando pessoas com a mesma quantidade de fatores de risco e tirando-se a media de casos de AVC por grupo.

Conclui-se que essa visualização dos dados confirma uma noção quase que óbvia , onde claramente ao se acumular muitos fatores de risco a chance de alguém desenvolver mais um , nesse contexto o AVC, também aumenta.

Entretanto , vale destacar o comportamento desse crescimento no gráfico. O aumento da taxa de AVC não ocorre de forma linear, percebe-se que há picos de crescimento nas variações para 2 e 4 fatores de risco.

Separação dos dados de teste ,exclusão de atributos redundantes ou inuteis e substituição de valores faltantes numéricos pela mediana :

In [13]:


from sklearn.model_selection import train_test_split
#drop no atributo id foi feita por não contribuir para a capacidade de generalização do modelo

X = df.drop(columns=["stroke","id"])
y = df["stroke"]


#stratify utilizado no split  pelo fato da distribuicao ser extremamente assimetrica quianto a quantidade de pessoas com avc

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#BMI é o unico atributo que possui pelo menos um valor nulo.Escolha de substituição do valor nulo:mediana dos valores de bmi dos dados de treino(evita outliers)
median_bmi = X_train["bmi"].median()

X_train["bmi"] = X_train["bmi"].fillna(median_bmi)
X_test["bmi"] = X_test["bmi"].fillna(median_bmi)








Encoding dos valores categoricos e aplicação de scaling :

In [14]:
#One hot encoding para os valores categoricos
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)


X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)



#scaling
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "age",
    "avg_glucose_level",
    "bmi"
]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)


Escolha das métricas comparativas : Recall e ROC-AUC

--A escolha por recall e ROC-AUC deve-se ao forte desbalanceamento do dataset, em que apenas cerca de 4% dos indivíduos sofreram avc. Nesse contexto, métricas como acurácia podem ser enganosas, pois um modelo que classificasse quase todos os pacientes como saudáveis ainda obteria um resultado elevado.A escolha de metricas como precisao e f1 não foi concretizada por também serem afetadas pelo desbalanceamento dos dados do dataset .

--O Recall foi escolhido por medir a quantidade de casos de avc  corretamente identificados pelo modelo , o que para um contexto médico como no caso desse dataset selecionado, funciona muito bem por evitar falhas na detecção de pacientes de alto risco ,o que poderia causar problemas clínicos sérios .

--O ROC-AUC é a principal métrica para esse modelo  por ser robusto perante aos dados desbalanceados.Ele mede a habilidade do modelo de associar pacientes positivos para avc como pacientes de prioridade de risco em relação aos classificados como negativos para avc , portanto para um contexto médico onde precisaríamos ordenar aqueles pacientes por prioridade de risco , um modelo com um ROC-AUC alto representaria uma boa capacidade de discriminação entre indivíduos de alto e baixo risco .

TREINAMENTO DOS MODELOS :

Modelo Baseline escolhido : Logistic Regression

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# Treinamento
model = LogisticRegression(
    C=10000,
    max_iter=10000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# Predições
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]


results = pd.DataFrame({
    "Recall": [recall_score(y_test, pred)],
    "ROC-AUC": [roc_auc_score(y_test, prob)],

})

results


,Recall,ROC-AUC
0,0.8,0.843848


O modelo de Regressão Logística apresentou ROC-AUC de 0.84, indicando uma capacidade sólida  de discriminar pacientes de alto e baixo risco. O Recall obtido demonstra que o modelo consegue identificar uma parcela significativa dos pacientes que sofreram AVC, característica desejável para aplicações médicas.

Teste de um modelo mais complexo :Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# Treinamento
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Probabilidades
rf_prob = rf.predict_proba(X_test)[:, 1]

# Threshold(melhor foi o de 0.05)
threshold = 0.05
rf_pred = (rf_prob >= threshold).astype(int)

# Métricas
results_rf = pd.DataFrame({
    "Recall": [recall_score(y_test, rf_pred)],
    "ROC-AUC": [roc_auc_score(y_test, rf_prob)]
})


results_rf

,Recall,ROC-AUC
0,0.76,0.785093


Apesar de possuir métricas positivas , o RandomForest não apresentou uma performance preditiva superior às métricas registradas pela Regressão Logística , mesmo após tentativas de ajustes nos limiares.

Relatório de tentativas de melhorias técnicas:

Tentativas de melhoramento das predições dos modelos:
--Os modelos gradient boost e random forest foram testados, mas suas metricas não superaram o modelo do baseline.Uma possível explicação para o resultado está relacionada à dificuldade de modelos mais complexos de capturarem padrões robustos de classes minoritárias, onde para esse trabalho , seria a classe pouco numerosa dos pacientes positivos para AVC.

Análise de feature engineering:
--adição da feature :'age glucose'(multiplicação dos valores do atributos age e glucose level -->lógica seria aumentar a gravidade de um nivel elevado de glicose ao multiplicar com o valor de sua  idade, pois idade elevada seria um fator agravante à condição de alta glicose ).Tal adição não resultou em melhoras signifcativas nas metricas do modelo .

--adição da feature :'vascular risk' , que funcionava de uma maneira parecida com a feature acima ,mas que usava os valores dos atributos de hypertension e heart desease.


--A falta de sucesso na adição dessas features pode talvez ser explicada pelo fato dos valores do dataframe  original já possuírem a informação dessas novas features , então a adição dessas novas informações não auxiliou o modelo a ser mais preciso pois não houve uma adição de informação útil e nova para a corretude das métricas de previsão do modelo




Conclusão

A análise de graficos(visualização) confirmou a importância de fatores já reconhecidos pela literatura médica como  hipertensão, doenças cardíacas e níveis elevados de glicose. Além disso, observou-se que o acúmulo desses fatores está associado a um aumento significativo na incidência de AVC dentro do conjunto de dados analisado.

Como principal limitação do estudo, destaca-se o forte desbalanceamento da variável alvo e a quantidade relativamente reduzida de casos positivos presentes no dataset. Essas características dificultam a obtenção de classificações perfeitamente precisas e limitam o desempenho máximo alcançável pelos modelos.

Apesar das limitações observadas, os resultados indicam que os modelos foram capazes de capturar padrões relevantes presentes nos dados e produzir estimativas de risco úteis para a diferenciação entre pacientes de maior e menor probabilidade de ocorrência de AVC.